# Online Retail — Querying and Data Mining

This notebook analyses the cleaned purchase data that the ETL pipeline loads into `data/processed/Invoice_Records.db` (table `Purchase_transactions`). Run `uv run retail-etl` first to create or refresh the database.

| Section | Question |
| --- | --- |
| 1. Revenue by country | Which markets generate the most revenue? |
| 2. Country extract | What does the transaction data of one market look like? |
| 3. Market basket analysis | Which products are bought together? |
| 4. RFM segmentation | Which customers are the most valuable, and which are at risk? |

All data access and analysis functions come from the `retail_pipeline` package, so the notebook and the tested code use the same logic.

In [1]:
import pandas as pd

from retail_pipeline.analytics import build_basket, mine_rules, rfm_table
from retail_pipeline.load import query, transactions_by_country

pd.set_option("display.max_colwidth", 60)

## 1. Revenue by Country

Revenue is calculated as `Quantity × UnitPrice` on clean purchase lines. Cancellations and non-product lines were removed by the pipeline.

In [2]:
query(
    """
    SELECT Country,
           COUNT(DISTINCT InvoiceNo)            AS invoices,
           COUNT(DISTINCT CustomerID)           AS customers,
           ROUND(SUM(Quantity * UnitPrice), 2)  AS revenue
    FROM Purchase_transactions
    GROUP BY Country
    ORDER BY revenue DESC
    LIMIT 10
    """
)

,Country,invoices,customers,revenue
0,Germany,443,94,205381.15
1,France,379,87,183801.94
2,Spain,88,30,55706.56
3,Switzerland,47,21,52441.95
4,Sweden,34,8,36828.83
5,Italy,35,14,15820.24
6,Greece,5,4,4425.52
7,Iceland,7,1,4310.00


## 2. Country Extract: Germany

`transactions_by_country` runs a parameterised SQL query (`WHERE Country = ?`) and returns the result as a DataFrame. Germany is the largest market in this dataset and is the basis for the basket analysis below.

In [3]:
germany = transactions_by_country("Germany")
print(f"{len(germany)} records")
germany.head()

8643 records


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536527,22809,SET OF 6 T-LIGHTS SANTA,6,2010-12-01 13:04:00,2.95,12662,Germany
1,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-01 13:04:00,2.55,12662,Germany
2,536527,84945,MULTI COLOUR SILVER T-LIGHT HOLDER,12,2010-12-01 13:04:00,0.85,12662,Germany
3,536527,22242,5 HOOK HANGER MAGIC TOADSTOOL,12,2010-12-01 13:04:00,1.65,12662,Germany
4,536527,22244,3 HOOK HANGER MAGIC GARDEN,12,2010-12-01 13:04:00,1.95,12662,Germany


## 3. Market Basket Analysis (Apriori)

Association rule mining finds products that frequently appear on the same invoice. The steps are:

1. Group the lines by `InvoiceNo` and `Description` and sum `Quantity`.
2. Pivot the result: one row per invoice, one column per product.
3. One-hot encode: `True` if the product appears on the invoice.
4. Run Apriori to find frequent itemsets, then derive association rules.

A minimum support of 5% keeps only itemsets that appear on at least 1 in 20 invoices.

In [4]:
basket = build_basket(germany)
print(f"Basket: {basket.shape[0]} invoices x {basket.shape[1]} items")
basket.iloc[:5, :5]

Basket: 443 invoices x 1692 items


Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND
InvoiceNo,,,,,
536527,False,False,False,False,False
536840,False,False,False,False,False
536861,False,False,False,False,False
536967,False,False,False,False,False
536983,False,False,False,False,False


In [5]:
rules = mine_rules(basket, min_support=0.05, min_lift=1.0)
print(f"{len(rules)} rules")
rules.head(10).round(3)

18 rules


,antecedents,consequents,support,confidence,lift
10,RED RETROSPOT CHARLOTTE BAG,WOODLAND CHARLOTTE BAG,0.061,0.844,6.445
11,WOODLAND CHARLOTTE BAG,RED RETROSPOT CHARLOTTE BAG,0.061,0.466,6.445
1,PLASTERS IN TIN CIRCUS PARADE,PLASTERS IN TIN WOODLAND ANIMALS,0.070,0.585,4.113
0,PLASTERS IN TIN WOODLAND ANIMALS,PLASTERS IN TIN CIRCUS PARADE,0.070,0.492,4.113
6,PLASTERS IN TIN SPACEBOY,PLASTERS IN TIN WOODLAND ANIMALS,0.063,0.571,4.018
7,PLASTERS IN TIN WOODLAND ANIMALS,PLASTERS IN TIN SPACEBOY,0.063,0.444,4.018
13,ROUND SNACK BOXES SET OF 4 FRUITS,ROUND SNACK BOXES SET OF4 WOODLAND,0.135,0.833,3.296
12,ROUND SNACK BOXES SET OF4 WOODLAND,ROUND SNACK BOXES SET OF 4 FRUITS,0.135,0.536,3.296
15,SPACEBOY LUNCH BOX,ROUND SNACK BOXES SET OF4 WOODLAND,0.072,0.681,2.693
14,ROUND SNACK BOXES SET OF4 WOODLAND,SPACEBOY LUNCH BOX,0.072,0.286,2.693


**How to read a rule:** *antecedents → consequents*.

- **Support:** the share of all invoices that contain both sides.
- **Confidence:** the share of invoices containing the antecedent that also contain the consequent.
- **Lift:** how much more often the two sides occur together than they would by chance. A lift above 1 indicates a real association.

The strongest rules link products from the same design range, for example the *Charlotte bag* and the *Plasters in tin* collections. They are good candidates for bundles, "frequently bought together" recommendations and joint product placement.

## 4. RFM Customer Segmentation (All Countries)

Each customer is scored from 1 (lowest) to 4 (highest) on three dimensions, using quartiles of the customer base:

- **Recency:** days since the last purchase (lower is better).
- **Frequency:** number of distinct invoices.
- **Monetary:** total spend.

The sum of the three scores (3–12) assigns the customer to a segment: Champions (10 or more), Loyal (8–9), Potential (6–7) and At risk (5 or less).

In [6]:
sales = query("SELECT * FROM Purchase_transactions")
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])
rfm = rfm_table(sales)
rfm.head()

,Recency,Frequency,Monetary,R,F,M,Segment
CustomerID,,,,,,,
12347,2,7,4310.00,4,4,4,Champions
12349,19,1,1457.55,3,1,3,Potential
12354,232,1,1079.40,1,1,3,At risk
12357,33,1,6207.67,3,1,4,Loyal
12371,60,1,1527.96,2,1,3,Potential


In [7]:
rfm.groupby("Segment").agg(
    customers=("Recency", "size"),
    avg_recency_days=("Recency", "mean"),
    avg_orders=("Frequency", "mean"),
    avg_spend=("Monetary", "mean"),
).round(1).sort_values("avg_spend", ascending=False)

,customers,avg_recency_days,avg_orders,avg_spend
Segment,,,,
Champions,78,17.2,9.0,5014.2
Loyal,42,40.0,3.1,1724.6
Potential,66,89.9,1.9,1022.4
At risk,73,192.0,1.1,379.4


**Interpretation and actions:**

| Segment | Profile | Recommended action |
| --- | --- | --- |
| Champions | Bought recently, order often, spend the most | Retain and reward: VIP offers, early access to new products |
| Loyal | Regular customers with solid spend | Loyalty programme, upselling to higher-value products |
| Potential | Few orders, moderate spend | Encourage repeat purchases with personalised recommendations |
| At risk | Last purchase long ago, low frequency and spend | Win-back campaigns, such as reminder e-mails and time-limited discounts |